# Feature Engineering
This notebook handles feature engineering for the pain medication effectiveness prediction model, including creating target variables, encoding categorical features, and extracting text features.

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 2. Load Cleaned Data

In [ ]:
# TODO: Load cleaned data
cleaned_data_path = '../data/cleaned/pain_meds_cleaned.csv'

df = pd.read_csv(cleaned_data_path)

print(f"Cleaned data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
display(df.head())

## 3. Create Effectiveness Label (Target Variable)

In [ ]:
# TODO: Create function to categorize effectiveness based on rating
# Rating 8-10: Effective
# Rating 5-7: Partially Effective
# Rating 1-4: Not Effective

def create_effectiveness_label(rating):
    """
    Categorizes medication effectiveness based on rating.
    
    Args:
        rating: Numeric rating (1-10)
    
    Returns:
        Effectiveness category (Effective, Partially Effective, Not Effective)
    """
    if rating >= 8:
        return 'Effective'
    elif rating >= 5:
        return 'Partially Effective'
    else:
        return 'Not Effective'

# Apply function to create target variable
if 'rating' in df.columns:
    df['effectiveness'] = df['rating'].apply(create_effectiveness_label)
    
    print("Effectiveness label created!")
    print("\nEffectiveness Distribution:")
    print(df['effectiveness'].value_counts())
    print("\nPercentage Distribution:")
    print(df['effectiveness'].value_counts(normalize=True) * 100)
else:
    print("Warning: 'rating' column not found!")

## 4. Apply Function to Create Target Variable

In [ ]:
# TODO: Create numeric encoding for target variable (for classification)
# Map effectiveness labels to numeric values
effectiveness_mapping = {
    'Not Effective': 0,
    'Partially Effective': 1,
    'Effective': 2
}

if 'effectiveness' in df.columns:
    df['effectiveness_encoded'] = df['effectiveness'].map(effectiveness_mapping)
    
    print("Target variable encoding complete!")
    print("\nEncoded Distribution:")
    print(df['effectiveness_encoded'].value_counts().sort_index())
    
    # Verify mapping
    print("\nSample mapping:")
    display(df[['rating', 'effectiveness', 'effectiveness_encoded']].sample(10))

## 5. One-Hot Encoding for Categorical Variables

In [ ]:
# TODO: Apply one-hot encoding to drugName and condition
# Limit to top N categories to avoid too many features

print("Applying one-hot encoding...")

# Get top 30 drugs and conditions to reduce dimensionality
top_drugs = df['drugName'].value_counts().head(30).index
top_conditions = df['condition'].value_counts().head(20).index

# Create binary columns for top drugs
df['drugName_top'] = df['drugName'].apply(lambda x: x if x in top_drugs else 'Other')
drug_dummies = pd.get_dummies(df['drugName_top'], prefix='drug')

# Create binary columns for top conditions
df['condition_top'] = df['condition'].apply(lambda x: x if x in top_conditions else 'other')
condition_dummies = pd.get_dummies(df['condition_top'], prefix='condition')

print(f"\nDrug features created: {drug_dummies.shape[1]}")
print(f"Condition features created: {condition_dummies.shape[1]}")

# Combine with original dataframe
df_encoded = pd.concat([df, drug_dummies, condition_dummies], axis=1)

print(f"\nNew dataset shape: {df_encoded.shape}")

## 6. Extract Text Features from Reviews

In [ ]:
# TODO: Extract text-based features from review column if available

if 'review' in df_encoded.columns:
    print("Extracting text features from reviews...")
    
    # Feature 1: Review length (character count)
    df_encoded['review_length'] = df_encoded['review'].astype(str).apply(len)
    
    # Feature 2: Word count
    df_encoded['review_word_count'] = df_encoded['review'].astype(str).apply(lambda x: len(x.split()))
    
    # Feature 3: Presence of positive keywords
    positive_keywords = ['effective', 'works', 'great', 'excellent', 'relief', 'helped', 'better']
    df_encoded['has_positive_keywords'] = df_encoded['review'].astype(str).str.lower().apply(
        lambda x: any(keyword in x for keyword in positive_keywords)
    ).astype(int)
    
    # Feature 4: Presence of negative keywords
    negative_keywords = ['ineffective', 'useless', 'terrible', 'worse', 'pain', 'side effects', 'stopped']
    df_encoded['has_negative_keywords'] = df_encoded['review'].astype(str).str.lower().apply(
        lambda x: any(keyword in x for keyword in negative_keywords)
    ).astype(int)
    
    # Feature 5: Average word length
    df_encoded['avg_word_length'] = df_encoded['review'].astype(str).apply(
        lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0
    )
    
    print("Text features extracted!")
    print("\nText Feature Summary:")
    text_features = ['review_length', 'review_word_count', 'has_positive_keywords', 
                     'has_negative_keywords', 'avg_word_length']
    display(df_encoded[text_features].describe())
else:
    print("No review column found. Skipping text feature extraction.")

## 7. Feature Scaling (Optional)

In [ ]:
# TODO: Apply feature scaling to numeric features if needed
# Note: Tree-based models (like Random Forest) don't require scaling
# But it's useful for other algorithms

print("Applying feature scaling to numeric features...")

# Identify numeric columns to scale (exclude target and ID columns)
exclude_cols = ['effectiveness', 'effectiveness_encoded', 'rating', 'drugName', 
                'condition', 'drugName_top', 'condition_top', 'review']
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns
cols_to_scale = [col for col in numeric_cols if col not in exclude_cols and not col.startswith('drug_') 
                 and not col.startswith('condition_')]

if len(cols_to_scale) > 0:
    # Create a copy for scaled features
    scaler = StandardScaler()
    
    # Scale the features
    df_encoded[cols_to_scale] = scaler.fit_transform(df_encoded[cols_to_scale])
    
    print(f"Scaled {len(cols_to_scale)} numeric features:")
    print(cols_to_scale)
    print("\nScaled features summary:")
    display(df_encoded[cols_to_scale].describe())
else:
    print("No numeric features to scale.")

## 8. Save Processed Data

In [ ]:
# TODO: Save ML-ready dataset
output_path = '../data/processed/pain_meds_ml_ready.csv'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save to CSV
df_encoded.to_csv(output_path, index=False)

print(f"Processed data saved to: {output_path}")
print("\n" + "=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"Original features: {df.shape[1]}")
print(f"Final features: {df_encoded.shape[1]}")
print(f"Total records: {len(df_encoded):,}")
print(f"\nTarget variable: effectiveness_encoded")
print(f"Feature categories created:")
print(f"  - Drug encoding: {drug_dummies.shape[1]} features")
print(f"  - Condition encoding: {condition_dummies.shape[1]} features")
if 'review' in df.columns:
    print(f"  - Text features: {len(text_features)} features")
print("=" * 60)

# Display sample of final dataset
print("\nSample of processed data:")
display(df_encoded.head())